# ScanDar on Colab

Training moves between a local RTX 3060 and Colab depending on what is available, so nothing in the codebase assumes either one. This notebook makes a Colab runtime look like the local machine:

1. check what GPU we were given
2. mount Drive, so **checkpoints survive a session timeout**
3. clone the repo and install it
4. point `SCANDAR_DATA` / `SCANDAR_OUT` at Drive
5. run the sanity checks

After that, every other notebook and every `python train.py ...` command runs unchanged.

> **One-time setup:** copy `data/` to `MyDrive/scandar/data`. The source images are small (~150 MB with the backgrounds); the frozen evaluation sets add ~220 MB on top and can either be copied or regenerated in step 5, which takes a few minutes and produces byte-identical files.

In [ ]:
# 1. what did we get?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Drive — checkpoints and figures live here so a timeout costs nothing
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/scandar'

In [ ]:
# 3. code
REPO_URL = 'https://github.com/SepehrGhr/ScanDar.git'

import os
if not os.path.isdir('/content/ScanDar'):
    !git clone $REPO_URL /content/ScanDar
%cd /content/ScanDar
!git pull --ff-only

# Colab already ships torch + CUDA, so requirements.txt deliberately does not pin them.
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# 4. point the project at Drive — this is the whole portability story
import os
os.environ['SCANDAR_DATA'] = f'{DRIVE_ROOT}/data'
os.environ['SCANDAR_OUT'] = f'{DRIVE_ROOT}/outputs'

import scandar
print('data   ->', scandar.paths.data)
print('output ->', scandar.paths.out)

In [ ]:
# 5. verify before spending GPU time on a broken setup
!python scripts/prepare_data.py
!python scripts/freeze_eval_sets.py   # idempotent: free if the sets came across in Drive
!python scripts/sanity_checks.py

## Training here

```bash
python train.py --config configs/enhance.yaml
```

Every run checkpoints into `SCANDAR_OUT` with its optimiser, scaler and RNG state, so a session that dies mid-epoch resumes with `train.resume=auto` (the default) — **re-running the exact same command is the way to resume**. `train.max_hours` stops a run cleanly before Colab stops it messily:

```bash
python train.py --config configs/enhance.yaml --set train.max_hours=3
```

**Expect Colab to be the slower machine for this project**, which inverts the usual advice. The bottleneck is not the GPU — it is the CPU compositing and degrading synthetic photos, and a Colab runtime has about two cores against the development laptop's sixteen. Watch the `samples/s` figure the trainer prints; the laptop manages about 34.

Because of that, a bigger batch will not buy throughput here. `batch_size` and `grad_accum` are separate keys so that the *effective* batch can be held constant on a machine where the real one does not fit — at 256x256 patches the default batch of 16 peaks around 3 GB, so it fits everywhere and neither key needs changing. They exist for the day a model or a resolution does not.